# Ecobici Dataset Data Augmentation

In [ ]:
import pandas as pd
import numpy as np
import urllib.request

## Connection OSRM (Open Source Routing Machine)

In [14]:
url = "http://download.geofabrik.de/north-america/mexico-latest.osm.pbf"
print("Downloading Mexico map... (this is a large file, please wait a few minutes)")
urllib.request.urlretrieve(url, "mexico-latest.osm.pbf")
print("Download complete!")

Download complete!


In [16]:
!docker run -t -v "%cd%":/data osrm/osrm-backend osrm-extract -p /opt/bicycle.lua /data/mexico-latest.osm.pbf
!docker run -t -v "%cd%":/data osrm/osrm-backend osrm-partition /data/mexico-latest.osrm
!docker run -t -v "%cd%":/data osrm/osrm-backend osrm-customize /data/mexico-latest.osrm

[info] Parsed 0 location-dependent features with 0 GeoJSON polygons
[info] Using script /opt/bicycle.lua
[info] Input file: mexico-latest.osm.pbf
[info] Profile: bicycle.lua
[info] Threads: 22
[info] Parsing in progress..
[info] input file generated by osmium/1.16.0
[info] timestamp: 2026-11-19T21:22:20Z
[info] Using profile api version 4
[info] Parse relations ...
[info] Parse ways and nodes ...
[info] Using profile api version 4
[info] Using profile api version 4
[info] Using profile api version 4
[info] Using profile api version 4
[info] Using profile api version 4
[info] Using profile api version 4
[info] Using profile api version 4
[info] Using profile api version 4
[info] Using profile api version 4
[info] Using profile api version 4
[info] Using profile api version 4
[info] Using profile api version 4
[info] Using profile api version 4
[info] Using profile api version 4
[info] Using profile api version 4
[info] Using profile api version 4
[info] Using profile api version 4
[info

Unable to find image 'osrm/osrm-backend:latest' locally
latest: Pulling from osrm/osrm-backend
4c20f809069e: Pulling fs layer
e73ad3e5bcf3: Pulling fs layer
aed007321795: Pulling fs layer
69f309e04f11: Pulling fs layer
595e916ac67a: Pulling fs layer
4c20f809069e: Download complete
595e916ac67a: Download complete
e73ad3e5bcf3: Download complete
69f309e04f11: Download complete
aed007321795: Download complete
595e916ac67a: Pull complete
aed007321795: Pull complete
69f309e04f11: Pull complete
4c20f809069e: Pull complete
e73ad3e5bcf3: Pull complete
Digest: sha256:af5d4a83fb90086a43b1ae2ca22872e6768766ad5fcbb07a29ff90ec644ee409
Status: Downloaded newer image for osrm/osrm-backend:latest


[info] Computing recursive bisection
[info] Loaded compressed node based graph: 18989408 edges, 39372381 nodes
[info]  running partition: 128 1.2 0.25 10 1000 # max_cell_size balance boundary cuts small_component_size
[info] Found 32399092 SCC (12 large, 32399080 small)
[info] SCC run took: 2.61111s
[info] Full bisection done in 105.314s
[info] Loaded node based graph to edge based graph mapping
[info] Loaded edge based graph for mapping partition ids: 81213736 edges, 18953372 nodes
[info] Fixed 1465 unconnected nodes
[info] Edge-based-graph annotation:
[info]   level 1 #cells 80981 bit size 17
[info]   level 2 #cells 6910 bit size 13
[info]   level 3 #cells 441 bit size 9
[info]   level 4 #cells 19 bit size 5
[info] Renumbered data in 173.204 seconds
[info] MultiLevelPartition constructed in 5.44644 seconds
[info] CellStorage constructed in 1.81155 seconds
[info] MLD data writing took 9.26139 seconds
[info] Cells statistics per level
[info] Level 1 #cells 80981 #boundary nodes 2123537

In [17]:
import json

# Load the local JSON file
with open("stations.json", "r", encoding="utf-8") as file:
    bike_data = json.load(file)

# Access the list of stations
stations = bike_data["data"]["stations"]

# Example: Print the name and capacity of the first station
first_station = stations[0]
print(f"Station: {first_station['name']}, Capacity: {first_station['capacity']}")

Station: CE-710 Molino del Rey - Glorieta de la Lealtad, Capacity: 39


In [21]:
import time

!docker rm -f osrm-cdmx

# Start the server specifying the MLD algorithm
!docker run -d --name osrm-cdmx -p 5000:5000 -v "%cd%":/data osrm/osrm-backend osrm-routed --algorithm mld --max-table-size 1000 /data/mexico-latest.osrm

print("Waiting 15 seconds for the map to load into RAM...")
time.sleep(15)

# Verify it is running
!docker logs osrm-cdmx

osrm-cdmx
02b99f26a1d30aff45e44cd3db62abe8a6e68ec31c950eabdbab04d0282cfb36
Waiting 15 seconds for the map to load into RAM...
[info] starting up engines, v5.26.0
[info] Threads: 22
[info] IP address: 0.0.0.0
[info] IP port: 5000


In [22]:
!docker ps -a --filter "name=osrm-cdmx"
!docker logs osrm-cdmx

CONTAINER ID   IMAGE               COMMAND                  CREATED          STATUS          PORTS                                         NAMES
02b99f26a1d3   osrm/osrm-backend   "osrm-routed --algor…"   27 seconds ago   Up 26 seconds   0.0.0.0:5000->5000/tcp, [::]:5000->5000/tcp   osrm-cdmx
[info] starting up engines, v5.26.0
[info] Threads: 22
[info] IP address: 0.0.0.0
[info] IP port: 5000


In [24]:
import json
import requests
import numpy as np
import pandas as pd

# 1. Load your local stations.json file
with open("stations.json", "r", encoding="utf-8") as file:
    bike_data = json.load(file)

stations = bike_data["data"]["stations"]
n = len(stations)

station_ids = [s["station_id"] for s in stations]
coords = [f"{s['lon']},{s['lat']}" for s in stations]
coord_string = ";".join(coords)

# Pre-allocate empty NxN matrices
distances = np.zeros((n, n))
durations = np.zeros((n, n))

# 2. Query in batches of 100 source stations at a time
batch_size = 100
for start in range(0, n, batch_size):
    end = min(start + batch_size, n)
    sources = ";".join(str(i) for i in range(start, end))
    
    url = f"http://localhost:5000/table/v1/bicycle/{coord_string}?sources={sources}&annotations=duration,distance"
    res = requests.get(url).json()
    
    if res.get("code") == "Ok":
        distances[start:end, :] = res["distances"]
        durations[start:end, :] = res["durations"]
        print(f"Processed stations {start} to {end - 1}...")
    else:
        print(f"Error on batch {start}-{end}:", res)
        break

# 3. Convert to DataFrames and save to CSV
df_distances = pd.DataFrame(distances, index=station_ids, columns=station_ids)
df_durations = pd.DataFrame(durations, index=station_ids, columns=station_ids)

df_distances.to_csv("ecobici_distance_matrix_meters.csv")
df_durations.to_csv("ecobici_duration_matrix_seconds.csv")

print(f"\nDone! Final matrix shape: {df_distances.shape}")
display(df_distances.head())

Processed stations 0 to 99...
Processed stations 100 to 199...
Processed stations 200 to 299...
Processed stations 300 to 399...
Processed stations 400 to 499...
Processed stations 500 to 599...
Processed stations 600 to 676...

Done! Final matrix shape: (677, 677)


,1,5,6,7,8,9,10,11,12,14,...,686,694,695,697,698,699,700,701,702,705
1,0.0,8090.2,8482.6,4502.0,8997.9,6765.5,7091.0,6296.8,5402.5,3208.3,...,6343.2,7277.9,6965.6,6453.0,7369.0,7138.9,305.7,5163.0,7391.5,6121.7
5,7640.6,0.0,545.8,9391.9,1342.5,2214.1,2282.3,2178.7,3393.1,7308.2,...,4066.1,12373.8,3750.9,12168.7,4673.6,4559.2,7280.0,10878.6,12725.1,8217.3
6,8045.2,550.8,0.0,9796.4,846.0,2178.8,1856.9,2556.7,3771.1,7712.8,...,3640.7,12778.4,3325.5,12573.2,4282.1,4133.8,7658.0,11283.2,13129.7,8621.8
7,4777.6,9937.6,10333.7,0.0,10845.3,9618.4,10093.8,9283.9,8696.3,2501.6,...,9637.1,3673.3,10259.4,2895.9,10662.8,10432.8,4973.8,1605.9,3473.2,3398.1
8,8586.5,1092.1,638.6,10337.7,0.0,2633.8,2301.8,3098.1,4139.7,8254.1,...,3829.1,13319.7,3513.8,13114.5,4113.8,4322.1,8199.3,11824.5,13671.0,9163.1


In [25]:
df_distances.to_csv('station_distances.csv')